# Notebook 5 — Why does M4 use audio/video worse than dedicated models? (D1, D2; optional D3)

**Research question.** Notebook 4 found that dedicated video-only and audio+video models beat M4 evaluated with the same
modalities (video +2.41 pp, audio+video +1.98 pp macro-F1, all 3 seeds). Is the gap caused by
(a) M4's **representations** (the audio/video encoders carry less usable information),
(b) the **text-conditioned cross-attention** when text is absent, or
(c) M4's **downstream classifier / gate**, which was trained mostly on text-dominated inputs?

| Section | Question | What is trained |
|---|---|---|
| L | Can the MOSEI emotion labels be matched exactly to our clips? (verification only) | nothing |
| R | Reference baselines incl. the established M1+U3 late fusion | nothing (saved predictions) |
| D1 | How much sentiment information is **linearly** readable from each frozen representation? | linear probes (logistic regression) |
| D2 | Does a **new classifier head** on frozen M4/M3 representations close the gap? | small MLP heads, same architecture/training as the dedicated models |
| DEC | **Research decision** from pre-stated rules | nothing |
| EMO | Do emotion labels carry relatively more non-text signal than sentiment does? (secondary) | linear probes |
| D3 | *Optional, independently executable:* deterministic replay of M4 training with logging | **retrains M4** (must reproduce the v2 weights exactly) |

**Correction of earlier statements.**
1. *"No new models are trained"* was wrong. D1 trains linear probes, D2 trains classifier heads, EMO trains probes, and
   D3 retrains M4 (as a faithful replay). No **existing** model or artifact is changed.
2. *"Eq. 15 cannot be implemented because it needs emotion labels"* was wrong. EMO-FAR's Eq. 15 is
   L_align = Σ_i Σ_{m∈{a,v}} (1 − cos(z_t^i, z_m^i)), a representation-alignment loss added with weight λ1 (Eq. 21).
   It needs no labels. Our M2 omitted it; "M2 + Eq. 15" is added to the pending controlled-ablation list.
   Any use of the CMU-MOSEI emotion labels would be a **separate, emotion-supervised extension**, not Eq. 15.

**Status.** Exploratory: the MOSEI test set has been examined many times. Hyperparameters of probes/heads are chosen on
validation only. Any methodological claim needs confirmation on another dataset.

**Fast by default:** `FULL_CONTEXT = False` computes only what the decision rule needs (raw features, dedicated models, M4):
30 linear probes and 12 heads instead of 66 and 54. Set it to True to add the M2/M3 context rows.

**Progress and restarts:** D1 and D2 print progress and save every finished probe/head immediately (`D1_parts/`, `D2_parts/`);
after a restart, Run all again and the finished ones are skipped.

**Runtime:** CPU + High-RAM (same as v2). Estimated (not measured on MOSEI): Sections L–EMO about 30–45 minutes;
D3, if switched on, about 20–25 minutes more. D3 is off by default.

In [1]:
import os, sys, time, json, random, pickle, platform, hashlib, itertools
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
DEVICE = "cpu"
torch.backends.cudnn.deterministic = True

Mounted at /content/drive


## 1. Paths and settings

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
LABEL_CSV = "/content/drive/MyDrive/datasets/CMU-MOSEI/label.csv"
LABEL_CSD = "/content/drive/MyDrive/datasets/CMU-MOSEI/CMU_MOSEI_Labels.csd"
RUN_D3 = False                      # optional replay of M4 training (Section D3); see its description first
RUN_D1 = True                       # linear probes (fast PyTorch version of the base paper's Eq. 16 classifier)
RUN_EMO = False                     # secondary emotion check (also logistic regressions); run later if wanted.
FULL_CONTEXT = False                # False (fast): only what the decision needs (raw, dedicated U, M4).
                                    # True: also M2/M3 and text-present representations (context only, ~3x slower)
D3_SEEDS = [42, 43, 44]
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]
    LABEL_CSV = os.environ.get("EMOFAR_LABEL_CSV", "none"); LABEL_CSD = os.environ.get("EMOFAR_LABEL_CSD", "none")
    RUN_D3 = os.environ.get("EMOFAR_RUN_D3") == "1"
V2_DIR, V3_DIR, V4_DIR = (f"{BASE_OUT}/{d}" for d in ["mosei_v2_cpu", "mosei_v3_diagnostics", "mosei_v4_capacity_utility"])
OUT_DIR = f"{BASE_OUT}/mosei_v5_representation_diagnostics"
N_BOOT = 10000 if not TEST_MODE else 500
for d in [V2_DIR, V4_DIR]:
    assert os.path.isdir(d), f"missing folder: {d}"
assert os.path.abspath(OUT_DIR) not in {os.path.abspath(p) for p in [V2_DIR, V3_DIR, V4_DIR]}
os.makedirs(OUT_DIR, exist_ok=True)

v2_info = json.load(open(f"{V2_DIR}/run_info.json"))
SEEDS = v2_info["config"]["SEEDS"]
logs = [json.load(open(f"{V2_DIR}/checkpoints/{m}_seed{s}_log.json")) for m in ["M1", "M2", "M3", "M4"] for s in SEEDS]
cfg = logs[0]["train_config"]
assert all(l["train_config"] == cfg for l in logs)
DATA_PATH = cfg["DATA_PATH"] if not TEST_MODE else os.environ["EMOFAR_MOSI"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
THREADS = int(v2_info["environment"]["torch_threads"])
torch.set_num_threads(THREADS)
ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "torch_threads": torch.get_num_threads(), "os": platform.platform(),
       "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
try:
    ENV["cpu"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except Exception:
    ENV["cpu"] = platform.processor() or "unknown"
print(json.dumps(ENV, indent=1)); print("v2 environment:", v2_info["environment"])
json.dump({"environment": ENV, "v2_train_config": cfg, "seeds": SEEDS, "threads": THREADS, "n_boot": N_BOOT, "run_d3": RUN_D3},
          open(f"{OUT_DIR}/run_info.json", "w"), indent=2)
RUN_DIR = OUT_DIR                   # the unchanged v2 data cell saves labels/ids into RUN_DIR -> the new folder
CKPT_DIR = f"{V2_DIR}/checkpoints"
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]

{
 "python": "3.13.18",
 "torch": "2.14.1+cpu",
 "numpy": "2.4.6",
 "pandas": "2.2.3",
 "sklearn": "1.6.1",
 "torch_threads": 4,
 "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "started_utc": "2026-11-24 12:54:02",
 "cpu": "AMD EPYC 7B12"
}
v2 environment: {'python': '3.13.18', 'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'cpu': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'torch_threads': 4, 'os': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'started_utc': '2026-11-24 06:35:51'}


## 2. Data (code copied unchanged from v2) and exact agreement with the v2 artifacts

In [3]:
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")

def prep(split):
    d = raw[split]
    mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
    feats = {"text": d["text"].astype(np.float32),
             "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
             "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
    score = d["regression_labels"].astype(np.float32)
    y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
    return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

data = {s: prep(s) for s in ["train", "valid", "test"]}
del raw

for m in ["audio", "vision"]:                                           # standardise with train statistics
    tr = data["train"]["feats"][m][data["train"]["mask"]]
    mu, sd = tr.mean(0), tr.std(0) + 1e-6
    for s in data:
        x = (data[s]["feats"][m] - mu) / sd
        data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
print("Feature sizes:", DIMS)
for s in data:
    counts = np.bincount(data[s]["y"], minlength=3)
    print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
v2_test_ids = pd.read_csv(f"{V2_DIR}/test_ids.csv", dtype=str)["id"].values
assert (v2_test_ids == np.asarray(data["test"]["id"]).astype(str)).all(), "test IDs/order differ from v2"
assert (np.load(f"{V2_DIR}/test_labels.npy") == data["test"]["y"]).all(), "test labels differ from v2"
assert (np.load(f"{V2_DIR}/valid_labels.npy") == data["valid"]["y"]).all(), "validation labels differ from v2"
assert len(data["train"]["y"]) == cfg["n_train"], "training-set size differs from v2"
print("Data match the v2 artifacts exactly.")

Loaded in 48s
Feature sizes: {'text': 768, 'audio': 74, 'vision': 35}
train: 16,326 clips | negative 4,738 / neutral 3,540 / positive 8,048
valid:  1,871 clips | negative 506 / neutral 433 / positive 932
test :  4,659 clips | negative 1,350 / neutral 1,025 / positive 2,284
Data match the v2 artifacts exactly.


## 3. Model definitions (copied unchanged from v2)

In [4]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

## 4. Training/evaluation functions (copied unchanged from v2)

In [5]:
from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 5. Helpers: video-grouped bootstrap (same as Notebook 4)

In [6]:
def video_of(ids):
    return np.array([str(i).split("$_$")[0] for i in ids])

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

y_test, y_val = data["test"]["y"], data["valid"]["y"]
_, vidx_test = np.unique(video_of(data["test"]["id"]), return_inverse=True); G_TEST = vidx_test.max() + 1
_rng = np.random.RandomState(2026)
W_TEST = np.stack([np.bincount(_rng.randint(0, G_TEST, G_TEST), minlength=G_TEST) for _ in range(N_BOOT)]).astype(np.float32)
SEED_DRAWS = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))

def video_counts(pred):
    c = np.zeros((G_TEST, 9)); np.add.at(c, (vidx_test, y_test * 3 + pred), 1); return c

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def grouped_boot(preds_a, preds_b):
    per_seed = np.array([f1_score(y_test, a, average="macro") - f1_score(y_test, b, average="macro") for a, b in zip(preds_a, preds_b)])
    d = np.stack([f1_from_counts(W_TEST @ video_counts(a)) - f1_from_counts(W_TEST @ video_counts(b)) for a, b in zip(preds_a, preds_b)])
    video = d.mean(0); seed_video = d[SEED_DRAWS, np.arange(N_BOOT)[:, None]].mean(1)
    return {"mean_diff_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100*x:+.2f}" for x in per_seed),
            "seeds_positive": int((per_seed > 0).sum()), "seeds_negative": int((per_seed < 0).sum()),
            "video_lo_pp": 100 * np.percentile(video, 2.5), "video_hi_pp": 100 * np.percentile(video, 97.5), "video_p": _p(video),
            "seedvideo_lo_pp": 100 * np.percentile(seed_video, 2.5), "seedvideo_hi_pp": 100 * np.percentile(seed_video, 97.5),
            "seedvideo_p": _p(seed_video)}

def mf1(y, p):
    return f1_score(y, p, average="macro")

## L. Emotion-label verification (MOSEI only; nothing is trained, nothing is used downstream unless all checks pass)

`CMU_MOSEI_Labels.csd` (CMU Multimodal SDK, HDF5, sequence "All Labels") stores, per video, time intervals and 7 values:
sentiment (−3…+3) and the intensities of happy, sad, anger, surprise, disgust, fear (0…3).
`label.csv` (MMSA) stores `video_id, clip_id, text, label (sentiment), mode (split)`; its `label_T/A/V` columns are empty.
**Matching rule tested:** within a video, the k-th smallest `clip_id` corresponds to the k-th earliest interval.
Checks: (1) dimensions and value ranges; (2) every `aligned_50` clip ID in every split found exactly once in label.csv with
the same split; (3) duplicates, unmatched IDs, missing values; (4) sentiment in the .csd equals the `aligned_50`
regression label for every matched clip; (5) interval duration vs. word count (an independent plausibility check of the
ordering: the correct order should correlate strongly, a shuffled order much less).

In [7]:
EMO_NAMES = ["happy", "sad", "anger", "surprise", "disgust", "fear"]
emo_ok, emo_table = False, None
if os.path.exists(LABEL_CSV) and os.path.exists(LABEL_CSD):
    import h5py
    lab = pd.read_csv(LABEL_CSV)
    h5 = h5py.File(LABEL_CSD, "r"); seq = h5[list(h5.keys())[0]]
    dim_names = json.loads(seq["metadata"]["dimension names"][0].decode()) if "dimension names" in seq["metadata"] else None
    print("csd sequence:", list(h5.keys())[0], "| dimension names:", dim_names, "| videos:", len(seq["data"]))
    report = {"dimension_names": dim_names, "label_csv_rows": len(lab),
              "duplicate_ids_label_csv": int(lab.duplicated(["video_id", "clip_id"]).sum()),
              "missing_sentiment_label_csv": int(lab["label"].isna().sum()),
              "label_csv_videos_absent_from_csd": int(len(set(lab.video_id) - set(seq["data"].keys())))}
    rows, count_mismatch = [], []
    for vid, grp in lab.groupby("video_id"):
        if vid not in seq["data"]:
            continue
        feats = seq["data"][vid]["features"][()]; iv = seq["data"][vid]["intervals"][()]
        if len(grp) != len(feats):
            count_mismatch.append((vid, len(grp), len(feats))); continue
        order = np.argsort(iv[:, 0], kind="stable"); g = grp.sort_values("clip_id")
        for r, k in zip(g.itertuples(), order):
            rows.append((f"{r.video_id}$_${r.clip_id}", r.mode, r.label, *feats[k], iv[k, 0], iv[k, 1], len(str(r.text).split())))
    emo = pd.DataFrame(rows, columns=["id", "mode", "label_csv_sentiment", "csd_sentiment", *EMO_NAMES, "start", "end", "n_words"])
    report["videos_with_count_mismatch"] = count_mismatch
    report["clips_unmatched_due_to_count_mismatch"] = int(sum(n for _, n, _ in count_mismatch))
    report["max_abs_csd_vs_labelcsv_sentiment"] = float((emo.csd_sentiment - emo.label_csv_sentiment).abs().max())
    report["value_ranges"] = emo[["csd_sentiment"] + EMO_NAMES].agg(["min", "max"]).round(4).to_dict()
    # agreement with aligned_50, per split
    emo_idx = emo.set_index("id")
    for split in ["train", "valid", "test"]:
        ids = np.asarray(data[split]["id"]).astype(str)
        found = np.isin(ids, emo_idx.index)
        lab_ids = set(lab[lab["mode"] == split].apply(lambda r: f"{r.video_id}$_${r.clip_id}", axis=1))
        report[f"{split}_n_pkl"] = len(ids)
        report[f"{split}_pkl_ids_in_label_csv_same_split"] = int(np.isin(ids, list(lab_ids)).sum())
        report[f"{split}_matched_to_csd"] = int(found.sum())
        report[f"{split}_unmatched_ids"] = ids[~found].tolist()
        report[f"{split}_max_abs_csd_vs_pkl_sentiment"] = float(np.abs(emo_idx.loc[ids[found], "csd_sentiment"].values - data[split]["score"][found]).max())
    dur = emo.end - emo.start
    shuffled = emo.assign(v=emo.id.str.split(r"\$_\$", regex=True).str[0], dur=dur).groupby("v")["dur"].transform(
        lambda x: x.sample(frac=1, random_state=0).values)
    report["spearman_duration_vs_words_matched"] = float(stats.spearmanr(dur, emo.n_words)[0])
    report["spearman_duration_vs_words_shuffled_within_video"] = float(stats.spearmanr(shuffled, emo.n_words)[0])
    emo_ok = (report["duplicate_ids_label_csv"] == 0 and report["missing_sentiment_label_csv"] == 0
              and all(report[f"{s}_pkl_ids_in_label_csv_same_split"] == report[f"{s}_n_pkl"] for s in ["train", "valid", "test"])
              and all(report[f"{s}_max_abs_csd_vs_pkl_sentiment"] < 1e-4 for s in ["train", "valid", "test"])
              and report["spearman_duration_vs_words_matched"] > 0.7)
    report["ALL_CHECKS_PASSED"] = bool(emo_ok)
    json.dump(report, open(f"{OUT_DIR}/label_verification.json", "w"), indent=2, default=str)
    emo.to_csv(f"{OUT_DIR}/mosei_emotion_labels_matched.csv", index=False)
    emo_table = emo_idx
    for k, v in report.items():
        print(f"  {k}: {v if not isinstance(v, list) or len(v) < 8 else str(v[:8]) + f' ... ({len(v)} items)'}")
else:
    print("Label files not found; Section L and the EMO section are skipped.")

csd sequence: All Labels | dimension names: ['sentiment', 'happy', 'sad', 'anger', 'surprise', 'disgust', 'fear'] | videos: 3293
  dimension_names: ['sentiment', 'happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']
  label_csv_rows: 22856
  duplicate_ids_label_csv: 0
  missing_sentiment_label_csv: 0
  label_csv_videos_absent_from_csd: 0
  videos_with_count_mismatch: [('JGEEA_JVriE', 1, 4), ('aa0J1AXSseY', 4, 7), ('zsRTbbKlsEg', 1, 2)]
  clips_unmatched_due_to_count_mismatch: 6
  max_abs_csd_vs_labelcsv_sentiment: 0.0
  value_ranges: {'csd_sentiment': {'min': -3.0, 'max': 3.0}, 'happy': {'min': 0.0, 'max': 3.0}, 'sad': {'min': 0.0, 'max': 3.0}, 'anger': {'min': 0.0, 'max': 3.0}, 'surprise': {'min': 0.0, 'max': 3.0}, 'disgust': {'min': 0.0, 'max': 3.0}, 'fear': {'min': 0.0, 'max': 1.6667}}
  train_n_pkl: 16326
  train_pkl_ids_in_label_csv_same_split: 16326
  train_matched_to_csd: 16324
  train_unmatched_ids: ['JGEEA_JVriE$_$0', 'zsRTbbKlsEg$_$6']
  train_max_abs_csd_vs_pkl_sentiment: 

## R. Reference baselines on the full test set (saved predictions only)
M1 (text only), M4, and the **late fusion M1+U3** (weighted average of M1 and the dedicated audio+video model U3;
weight chosen per seed on validation from 0, 0.05, …, 0.5). This baseline is kept in all future comparisons.

In [8]:
GRID_W = np.round(np.arange(0, 0.501, 0.05), 2)
ref_rows, ref_pred = [], {"M1": [], "M4": [], "LF M1+U3": []}
for s in SEEDS:
    m1v, m1t = np.load(f"{CKPT_DIR}/M1_seed{s}_valid_probs.npy"), np.load(f"{CKPT_DIR}/M1_seed{s}_test_probs.npy")
    u3v, u3t = np.load(f"{V4_DIR}/checkpoints/U3_seed{s}_valid_probs.npy"), np.load(f"{V4_DIR}/checkpoints/U3_seed{s}_test_probs.npy")
    m4t = np.load(f"{CKPT_DIR}/M4_seed{s}_test_probs.npy")
    w = max(GRID_W, key=lambda w: mf1(y_val, ((1 - w) * m1v + w * u3v).argmax(1)))
    lf = (1 - w) * m1t + w * u3t
    np.save(f"{OUT_DIR}/LF_M1_U3_seed{s}_test_probs.npy", lf.astype(np.float32))
    for k, p in [("M1", m1t), ("M4", m4t), ("LF M1+U3", lf)]:
        ref_pred[k].append(p.argmax(1)); r = evaluate(p, data["test"]); r.update(model=k, seed=s, lf_weight=w if k.startswith("LF") else np.nan); ref_rows.append(r)
ref = pd.DataFrame(ref_rows); ref.to_csv(f"{OUT_DIR}/R_reference_baselines.csv", index=False)
ref_cmp = pd.DataFrame([{"comparison": f"{a} − {b}", **grouped_boot(ref_pred[a], ref_pred[b])} for a, b in [("LF M1+U3", "M4"), ("LF M1+U3", "M1"), ("M4", "M1")]])
ref_cmp.to_csv(f"{OUT_DIR}/R_reference_comparisons.csv", index=False)
print(ref.groupby("model", sort=False)[["accuracy", "macro_f1", "acc2", "ece"]].agg(["mean", "std"]).round(4))
ref_cmp.round(4)

         accuracy         macro_f1            acc2             ece        
             mean     std     mean     std    mean     std    mean     std
model                                                                     
M1         0.6525  0.0088   0.6319  0.0037  0.8385  0.0046  0.0177  0.0048
M4         0.6619  0.0082   0.6345  0.0050  0.8369  0.0048  0.0206  0.0057
LF M1+U3   0.6615  0.0055   0.6386  0.0005  0.8422  0.0042  0.0723  0.0174


,comparison,mean_diff_pp,per_seed_pp,seeds_positive,seeds_negative,video_lo_pp,video_hi_pp,video_p,seedvideo_lo_pp,seedvideo_hi_pp,seedvideo_p
0,LF M1+U3 − M4,0.4157,"+0.08, +1.04, +0.13",3,0,-0.1398,0.9884,0.1342,-0.4449,1.3058,0.3256
1,LF M1+U3 − M1,0.6738,"+0.92, +0.31, +0.79",3,0,0.2221,1.1472,0.0060,0.0212,1.2703,0.0448
2,M4 − M1,0.2581,"+0.84, -0.73, +0.66",2,1,-0.3991,0.9090,0.4500,-0.8963,1.3560,0.6293


## 6. Representation extraction (frozen models, inference only)

For every seed, from the saved checkpoints:
- **raw** — masked mean over time of the standardised input features (no learning; seed-independent).
- **U (dedicated)** — the 128-d hidden layer (after ReLU) of the dedicated U1/U2/U3 models (Notebook 4).
- **M2 z** — EMO-FAR-RMSR projection z_m (Eq. 10), pooled.
- **M3/M4 pre** — the modality's own projected, normalised, pooled representation **before** cross-attention (does not depend on text).
- **M3/M4 post|alone** — the modality's representation **after** text-conditioned cross-attention, with text (and the other modality) hidden.
  This is exactly the vector M3/M4's own classifier receives in the single-modality condition.
- **M3/M4 post|full** — after cross-attention with text present (attention pooling chosen by text; values still from this modality only).
- For audio+video: **raw A+V**, **U3**, **M4 pre A ⊕ pre V** (concatenation), and **M4 fused|A+V** (the classifier input when text is hidden).

A check confirms that the extraction code reproduces each model's own outputs exactly.

In [9]:
@torch.no_grad()
def extract_fixed(model, d, avail_row, check=False):
    """Re-implements FixedFusion.forward step by step and returns intermediate representations."""
    model.eval(); n = len(d["y"]); out = {k: [] for k in ["pre_audio", "pre_vision", "post_audio", "post_vision", "fused", "logits"]}
    for i in range(0, n, 256):
        idx = np.arange(i, min(i + 256, n))
        x, mask, av = to_batch(d, idx, np.tile(np.asarray(avail_row, np.float32), (len(idx), 1)))
        p = {m: model.proj[m](x[m]) * av[:, j, None, None] for j, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            att, _ = model.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(att + p[m], mask)
            out[f"pre_{m}"].append(masked_mean(p[m], mask).numpy()); out[f"post_{m}"].append(h[m].numpy())
        lg = model.gate(torch.cat([h[m] for m in MODS], -1)).masked_fill(av == 0, float("-inf"))
        w = lg.softmax(-1); fused = sum(w[:, j:j + 1] * h[m] for j, m in enumerate(MODS))
        out["fused"].append(fused.numpy()); out["logits"].append(model.cls(fused).numpy())
        if check:
            assert torch.equal(model.cls(fused), model(x, mask, av)), "extraction does not reproduce the model output"
    return {k: np.concatenate(v) for k, v in out.items()}

@torch.no_grad()
def extract_m2(model, d):
    model.eval(); n = len(d["y"]); z = {m: [] for m in ["audio", "vision"]}
    for i in range(0, n, 256):
        idx = np.arange(i, min(i + 256, n)); x, mask, av = to_batch(d, idx, np.ones((len(idx), 3), np.float32))
        for m in z: z[m].append(model.proj[m](masked_mean(x[m], mask)).numpy())
    return {m: np.concatenate(v) for m, v in z.items()}

class PooledMLP(nn.Module):                       # identical to Notebook 4
    def __init__(self, mods):
        super().__init__()
        self.mods = list(mods)
        self.net = nn.Sequential(nn.Linear(sum(DIMS[m] for m in self.mods), D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(torch.cat([masked_mean(x[m], mask) for m in self.mods], -1))
        return (out, None) if return_gate else out

@torch.no_grad()
def extract_u(model, d):
    model.eval(); n = len(d["y"]); hid = []
    for i in range(0, n, 256):
        idx = np.arange(i, min(i + 256, n)); x, mask, av = to_batch(d, idx, np.ones((len(idx), 3), np.float32))
        hid.append(model.net[1](model.net[0](torch.cat([masked_mean(x[m], mask) for m in model.mods], -1))).numpy())
    return np.concatenate(hid)

def raw_pooled(d, mods):
    m = d["mask"][..., None].astype(np.float32)
    return np.concatenate([(d["feats"][k] * m).sum(1) / np.maximum(m.sum(1), 1) for k in mods], -1)

SPLITS = ["train", "valid", "test"]
REP = {}                                            # REP[(name, seed)][split] -> array; seed None = seed-independent
TARGET_OF = {}                                      # which modality condition each representation belongs to
for k, mods in [("raw audio", ["audio"]), ("raw vision", ["vision"]), ("raw A+V", ["audio", "vision"])]:
    REP[(k, None)] = {sp: raw_pooled(data[sp], mods) for sp in SPLITS}
TARGET_OF.update({"raw audio": "audio", "raw vision": "vision", "raw A+V": "A+V"})
own_pred = {}                                       # model's own single-modality predictions (test) for reference
t0 = time.time()
for s in SEEDS:
    print(f"Extracting representations for seed {s} ({time.time() - t0:.0f}s elapsed)", flush=True)
    for tag, mods, tgt in [("U1", ["audio"], "audio"), ("U2", ["vision"], "vision"), ("U3", ["audio", "vision"], "A+V")]:
        u = PooledMLP(mods); u.load_state_dict(torch.load(f"{V4_DIR}/checkpoints/{tag}_seed{s}.pt", map_location="cpu"))
        REP[(f"U dedicated {tgt}", s)] = {sp: extract_u(u, data[sp]) for sp in SPLITS}; TARGET_OF[f"U dedicated {tgt}"] = tgt
        own_pred[(f"U dedicated {tgt}", s)] = np.load(f"{V4_DIR}/checkpoints/{tag}_seed{s}_test_probs.npy").argmax(1)
    if FULL_CONTEXT:
        m2 = PaperRMSR(); m2.load_state_dict(torch.load(f"{CKPT_DIR}/M2_seed{s}.pt", map_location="cpu"))
        z = {sp: extract_m2(m2, data[sp]) for sp in SPLITS}
        REP[("M2 z audio", s)] = {sp: z[sp]["audio"] for sp in SPLITS}; REP[("M2 z vision", s)] = {sp: z[sp]["vision"] for sp in SPLITS}
        TARGET_OF.update({"M2 z audio": "audio", "M2 z vision": "vision"})
    conds = [("audio", (0, 1, 0)), ("vision", (0, 0, 1)), ("A+V", (0, 1, 1))] + ([("full", (1, 1, 1))] if FULL_CONTEXT else [])
    for M in (["M3", "M4"] if FULL_CONTEXT else ["M4"]):
        mdl = FixedFusion(); mdl.load_state_dict(torch.load(f"{CKPT_DIR}/{M}_seed{s}.pt", map_location="cpu"))
        ex = {(cond, sp): extract_fixed(mdl, data[sp], row, check=(sp == "test")) for cond, row in conds for sp in SPLITS}
        for m in ["audio", "vision"]:
            # the pre-attention representation does not depend on which modalities are available (checked below when possible)
            REP[(f"{M} pre {m}", s)] = {sp: ex[(m, sp)][f"pre_{m}"] for sp in SPLITS}
            REP[(f"{M} post|alone {m}", s)] = {sp: ex[(m, sp)][f"post_{m}"] for sp in SPLITS}
            TARGET_OF.update({f"{M} pre {m}": m, f"{M} post|alone {m}": m})
            if FULL_CONTEXT:
                REP[(f"{M} post|full {m}", s)] = {sp: ex[("full", sp)][f"post_{m}"] for sp in SPLITS}; TARGET_OF[f"{M} post|full {m}"] = m
            own_pred[(f"{M} own {m}", s)] = ex[(m, "test")]["logits"].argmax(1)
            assert np.allclose(ex[(m, "test")]["pre_" + m], ex[("A+V", "test")]["pre_" + m]), "pre-attention rep should not depend on availability"
        REP[(f"{M} pre A⊕V", s)] = {sp: np.hstack([ex[("audio", sp)]["pre_audio"], ex[("vision", sp)]["pre_vision"]]) for sp in SPLITS}
        REP[(f"{M} fused|A+V", s)] = {sp: ex[("A+V", sp)]["fused"] for sp in SPLITS}
        TARGET_OF.update({f"{M} pre A⊕V": "A+V", f"{M} fused|A+V": "A+V"})
        own_pred[(f"{M} own A+V", s)] = ex[("A+V", "test")]["logits"].argmax(1)
        if M == "M4":   # must equal the saved Notebook 3/4 subset predictions
            for m, key in [("vision", "V"), ("audio", "A"), ("A+V", "AV")]:
                f = f"{V4_DIR}/m4_subset_probs/M4_seed{s}_test_{key}.npy"
                if os.path.exists(f):
                    assert (np.load(f).argmax(1) == own_pred[(f"M4 own {m}", s)]).all(), f"M4 {m} predictions differ from Notebook 4"
print(f"Extracted {len(REP)} representation sets in {time.time() - t0:.0f}s; extraction reproduces model outputs exactly.")

def rep_seeds(name):
    return [None] if (name, None) in REP else SEEDS

def get_rep(name, s, split):
    return REP[(name, None)][split] if (name, None) in REP else REP[(name, s)][split]

Extracting representations for seed 42 (0s elapsed)
Extracting representations for seed 43 (41s elapsed)
Extracting representations for seed 44 (77s elapsed)
Extracted 30 representation sets in 113s; extraction reproduces model outputs exactly.


In [10]:
def train_head(Xtr, ytr, Xva, yva, Xte, seed, linear=False):
    set_seed(seed)
    sc = StandardScaler().fit(Xtr)
    Xtr_, Xva_, Xte_ = (torch.from_numpy(sc.transform(X).astype(np.float32)) for X in (Xtr, Xva, Xte))
    head = (nn.Linear(Xtr.shape[1], 3) if linear else
            nn.Sequential(nn.Linear(Xtr.shape[1], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3)))
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(ytr, minlength=3)
    cw = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float)
    rng = np.random.RandomState(seed); yt_ = torch.from_numpy(ytr)
    best, best_state, bad, ep_done = -1, None, 0, 0
    for ep in range(EPOCHS):
        head.train(); order = rng.permutation(len(ytr))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            loss = F.cross_entropy(head(Xtr_[idx]), yt_[idx], weight=cw)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step()
        head.eval()
        with torch.no_grad():
            val = mf1(yva, head(Xva_).argmax(1).numpy())
        ep_done = ep + 1
        if val > best:
            best, bad = val, 0; best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad():
        return best, ep_done, head(Xte_).argmax(1).numpy()

## D1. Linear probes — `RUN_D1`
**Probe = one fully connected layer + softmax**, the same classifier form as the base paper's Eq. 16, trained with the same
recipe as the D2 heads. (The earlier scikit-learn logistic-regression solver was far too slow on MOSEI and was replaced.)
D1 is context: the decision rule uses the D2 heads.
Multinomial logistic regression on standardised features, `class_weight="balanced"` (like the class-weighted training loss),
trained on the **training split**; the inverse regularisation C ∈ {0.001, 0.01, 0.1, 1, 10} is chosen by **validation**
macro-F1; the test set is scored once. A linear probe measures what is *linearly* readable, so it compares representations
with equal (minimal) readout capacity. Raw features have fewer dimensions (74 / 35 / 109) than learned ones (128 / 256);
this is noted, not corrected.

In [11]:
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]     # used only by the optional EMO section
N_JOBS = max(1, min(THREADS, os.cpu_count() or 1))
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
D1_PARTS = f"{OUT_DIR}/D1_linear_parts"; os.makedirs(D1_PARTS, exist_ok=True)
safe = lambda t: "".join(ch if ch.isalnum() else "_" for ch in t)

def probe_task(name, s, Xtr, ytr, Xva, yva, Xte):
    """Linear probe = one fully connected layer + softmax, i.e. the same classifier form as EMO-FAR's Eq. 16
    (y = softmax(W_c z + b_c)). Trained with the same recipe as the D2 heads (AdamW, class weights, early stopping on
    validation macro-F1). Fast: no slow solver, one thread per worker."""
    torch.set_num_threads(1)
    v, ep, pred = train_head(Xtr, ytr, Xva, yva, Xte, s if s is not None else SEEDS[0], linear=True)
    return name, s, (v, ep, pred)

if RUN_D1:
    tasks = [(name, s) for name in sorted({k for k, _ in REP}) for s in rep_seeds(name)]
    todo = [(n, s) for n, s in tasks if not os.path.exists(f"{D1_PARTS}/{safe(n)}_seed{s}.json")]
    print(f"D1: {len(tasks)} probes in total, {len(tasks) - len(todo)} already saved, {len(todo)} to run on {N_JOBS} cores", flush=True)
    t0 = time.time()
    for k in range(0, len(todo), 2 * N_JOBS):                         # small chunks: progress + saving after each chunk
        chunk = todo[k:k + 2 * N_JOBS]
        out = Parallel(n_jobs=N_JOBS)(delayed(probe_task)(n, s, get_rep(n, s, "train"), data["train"]["y"], get_rep(n, s, "valid"),
                                                          y_val, get_rep(n, s, "test")) for n, s in chunk)
        for n, s, (v, C, pred) in out:
            np.save(f"{D1_PARTS}/{safe(n)}_seed{s}_pred.npy", pred)
            json.dump({"representation": n, "seed": s, "stopped_epoch": C, "valid_macro_f1": v}, open(f"{D1_PARTS}/{safe(n)}_seed{s}.json", "w"))
        done = len(tasks) - len(todo) + k + len(chunk)
        print(f"  D1 progress: {done}/{len(tasks)} probes saved  ({time.time() - t0:.0f}s elapsed)", flush=True)

    d1_rows, d1_pred = [], {}
    for n, s in tasks:
        info = json.load(open(f"{D1_PARTS}/{safe(n)}_seed{s}.json")); pred = np.load(f"{D1_PARTS}/{safe(n)}_seed{s}_pred.npy")
        d1_pred[(n, s)] = pred
        d1_rows.append({"representation": n, "target": TARGET_OF[n], "seed": s, "stopped_epoch": info["stopped_epoch"], "valid_macro_f1": info["valid_macro_f1"],
                        "test_macro_f1": mf1(y_test, pred)})
    d1 = pd.DataFrame(d1_rows); d1.to_csv(f"{OUT_DIR}/D1_linear_probes_per_seed.csv", index=False)
    print(f"D1 complete ({time.time() - t0:.0f}s this session)")
    print(d1.groupby(["target", "representation"])[["valid_macro_f1", "test_macro_f1"]].agg(["mean", "std"]).round(4).to_string())
else:
    print("D1 skipped (RUN_D1 = False). The decision below uses D2 only.")

D1: 30 probes in total, 0 already saved, 30 to run on 4 cores
  D1 progress: 8/30 probes saved  (13s elapsed)
  D1 progress: 16/30 probes saved  (19s elapsed)
  D1 progress: 24/30 probes saved  (24s elapsed)
  D1 progress: 30/30 probes saved  (30s elapsed)
D1 complete (30s this session)
                            valid_macro_f1         test_macro_f1        
                                      mean     std          mean     std
target representation                                                   
A+V    M4 fused|A+V                 0.4175  0.0066        0.4250  0.0057
       M4 pre A⊕V                   0.4231  0.0026        0.4293  0.0012
       U dedicated A+V              0.4353  0.0054        0.4303  0.0075
       raw A+V                      0.4108     NaN        0.4246     NaN
audio  M4 post|alone audio          0.3900  0.0064        0.4011  0.0073
       M4 pre audio                 0.3977  0.0051        0.4041  0.0042
       U dedicated audio            0.4122  0.0058     

## D2. New classifier heads on frozen representations
Head = **the dedicated models' head**: Linear(d,128) → ReLU → Dropout 0.2 → Linear(128,3), trained with **the same**
optimiser, learning rate, weight decay, class weights, batch size, gradient clipping, maximum epochs and validation-based
early stopping as every v2/v4 model, seeds 42/43/44. The representation is frozen (pre-computed).
Note: an MLP head on the **raw** features is exactly the dedicated model U, so U's own predictions serve as that row.
**Interpretation rule (stated in advance):** if a new head on M4's frozen single-modality representation beats M4's own
prediction, that is evidence of a downstream (classifier/gate) limitation — not proof of a single cause.

In [12]:
D2_REPS = [n for n in sorted({k for k, _ in REP}) if not n.startswith("raw") and not n.startswith("U dedicated")]
if not FULL_CONTEXT:   # the pre-specified decision uses exactly these four
    D2_REPS = ["M4 post|alone vision", "M4 pre vision", "M4 fused|A+V", "M4 pre A⊕V"]
D2_PARTS = f"{OUT_DIR}/D2_parts"; os.makedirs(D2_PARTS, exist_ok=True)

def head_task(name, s, Xtr, ytr, Xva, yva, Xte):
    torch.set_num_threads(1)        # one thread per worker: deterministic, and 4 heads train at once
    return name, s, train_head(Xtr, ytr, Xva, yva, Xte, s)

tasks = [(name, s) for name in D2_REPS for s in SEEDS]
todo = [(n, s) for n, s in tasks if not os.path.exists(f"{D2_PARTS}/{safe(n)}_seed{s}.json")]
print(f"D2: {len(tasks)} heads in total, {len(tasks) - len(todo)} already saved, {len(todo)} to run on {N_JOBS} cores", flush=True)
t0 = time.time()
for k in range(0, len(todo), N_JOBS):
    chunk = todo[k:k + N_JOBS]
    out = Parallel(n_jobs=N_JOBS)(delayed(head_task)(n, s, get_rep(n, s, "train"), data["train"]["y"], get_rep(n, s, "valid"),
                                                     y_val, get_rep(n, s, "test")) for n, s in chunk)
    for n, s, (v, ep, pred) in out:
        np.save(f"{D2_PARTS}/{safe(n)}_seed{s}_pred.npy", pred)
        json.dump({"representation": n, "seed": s, "valid_macro_f1": v, "stopped_epoch": ep}, open(f"{D2_PARTS}/{safe(n)}_seed{s}.json", "w"))
    done = len(tasks) - len(todo) + k + len(chunk)
    print(f"  D2 progress: {done}/{len(tasks)} heads saved  ({time.time() - t0:.0f}s elapsed)", flush=True)

d2_rows, d2_pred = [], {}
for n, s in tasks:
    info = json.load(open(f"{D2_PARTS}/{safe(n)}_seed{s}.json")); pred = np.load(f"{D2_PARTS}/{safe(n)}_seed{s}_pred.npy")
    d2_pred[(n, s)] = pred
    d2_rows.append({"representation": n, "target": TARGET_OF[n], "seed": s, "valid_macro_f1": info["valid_macro_f1"],
                    "stopped_epoch": info["stopped_epoch"], "test_macro_f1": mf1(y_test, pred)})
for (name, s), pred in own_pred.items():         # reference rows: each model's own prediction
    d2_rows.append({"representation": name, "target": name.split()[-1],
                    "seed": s, "valid_macro_f1": np.nan, "stopped_epoch": np.nan, "test_macro_f1": mf1(y_test, pred)})
d2 = pd.DataFrame(d2_rows); d2.to_csv(f"{OUT_DIR}/D2_heads_per_seed.csv", index=False)
print(f"D2 complete ({time.time() - t0:.0f}s this session)")
d2.groupby(["target", "representation"])[["valid_macro_f1", "test_macro_f1"]].agg(["mean", "std"]).round(4)

D2: 12 heads in total, 0 already saved, 12 to run on 4 cores
  D2 progress: 4/12 heads saved  (6s elapsed)
  D2 progress: 8/12 heads saved  (11s elapsed)
  D2 progress: 12/12 heads saved  (16s elapsed)
D2 complete (16s this session)


valid_macro_f1         test_macro_f1        
                                      mean     std          mean     std
target representation                                                   
A+V    M4 fused|A+V                 0.4323  0.0078        0.4315  0.0047
       M4 own A+V                      NaN     NaN        0.4176  0.0042
       M4 pre A⊕V                   0.4323  0.0035        0.4355  0.0014
       U dedicated A+V                 NaN     NaN        0.4374  0.0042
audio  M4 own audio                    NaN     NaN        0.4002  0.0072
       U dedicated audio               NaN     NaN        0.4106  0.0010
vision M4 own vision                   NaN     NaN        0.3945  0.0117
       M4 post|alone vision         0.4316  0.0089        0.4239  0.0100
       M4 pre vision                0.4258  0.0036        0.4202  0.0051
       U dedicated vision              NaN     NaN        0.4186  0.0078

## DEC. Research decision (rules fixed before seeing the results)

For **video** and **audio+video** (audio is reported but its gap was not significant in Notebook 4):
- F_own = M4's own prediction with only those modalities; F_ded = dedicated model U; gap G = F_ded − F_own.
- First, the gap itself must be present in this run (U − M4 own: video-grouped CI above 0, all 3 seeds positive);
  otherwise there is nothing to explain and no flag is set.
- **Downstream (classifier/gate) limitation supported** if a new head on M4's frozen classifier input
  (`M4 post|alone` for video, `M4 fused|A+V` for audio+video) beats F_own with video-grouped 95 % CI above 0, all 3 seeds
  positive, and recovers ≥ 50 % of G.
- **Representation limitation supported** if, after the new head, the remaining gap to U is still ≥ 50 % of G with CI above 0
  and all 3 seeds positive — and the best M4 representation (pre or post, new head) also stays below U by ≥ 50 % of G.
- **Attention (text-absent pooling) limitation supported** if the new head on `M4 pre` beats the new head on `M4 post|alone`
  with CI above 0 and all 3 seeds positive (video only; for A+V: `M4 pre A⊕V` vs `M4 fused|A+V`).
- "Supported" means consistent with the evidence on this (re-used) test set, not established.

**What each outcome means for the project (also fixed in advance):**
- Representation limitation → run D3 (exposure, checkpoint selection, gradient diagnostics) before considering any
  balancing method; then compare only against established remedies (OGM-GE, MMPareto, Diagnosing & Re-learning,
  and the missing-rate-aware MCE).
- Downstream / attention limitation without representation limitation → encoder-balancing methods are **not** justified
  by our evidence; the problem is how rarely-seen input conditions are classified. Prior work on imbalanced missing
  rates and missing-modality MSA (MissBench 2026, MCE 2025, EBMC CVPR 2026, CMAD ICCV 2025) must be checked in detail
  before proposing anything.
- Nothing supported → the gap has no isolated mechanism at this scale; stop this line and report it as a limitation.

In [13]:
def compare(a_pred, b_pred):
    return grouped_boot([a_pred[s] for s in SEEDS], [b_pred[s] for s in SEEDS])

def preds_of(table_pred, name):
    return {s: table_pred[(name, s)] for s in SEEDS}

own = lambda name: {s: own_pred[(name, s)] for s in SEEDS}
dec_rows, decisions = [], {}
for tgt, own_name, ded_name, post_name, pre_name in [
        ("vision", "M4 own vision", "U dedicated vision", "M4 post|alone vision", "M4 pre vision"),
        ("A+V", "M4 own A+V", "U dedicated A+V", "M4 fused|A+V", "M4 pre A⊕V")]:
    F_own = np.mean([mf1(y_test, own_pred[(own_name, s)]) for s in SEEDS])
    F_ded = np.mean([mf1(y_test, own_pred[(ded_name, s)]) for s in SEEDS])
    G = 100 * (F_ded - F_own)
    c_gap = compare(own(ded_name), own(own_name))         # the gap itself must exist before it is explained
    head_post, head_pre = preds_of(d2_pred, post_name), preds_of(d2_pred, pre_name)
    c_head = compare(head_post, own(own_name))            # new head vs M4's own classifier (same input vector)
    c_rem = compare(own(ded_name), head_post)             # remaining gap after the new head
    c_att = compare(head_pre, head_post)                  # pre-attention vs post-attention representation
    val_mean = lambda nm: d2[(d2.representation == nm)].valid_macro_f1.mean()
    best_m4 = head_post if val_mean(post_name) >= val_mean(pre_name) else head_pre     # chosen on VALIDATION, not test
    c_best = compare(own(ded_name), best_m4)
    sig = lambda c: c["video_lo_pp"] > 0 and c["seeds_positive"] == len(SEEDS)
    gap_exists = c_gap["video_lo_pp"] > 0 and c_gap["seeds_positive"] == len(SEEDS)
    flags = {"gap_exists": bool(gap_exists),
             "downstream": bool(gap_exists and sig(c_head) and c_head["mean_diff_pp"] >= 0.5 * G),
             "representation": bool(gap_exists and sig(c_rem) and c_rem["mean_diff_pp"] >= 0.5 * G and sig(c_best) and c_best["mean_diff_pp"] >= 0.5 * G),
             "attention": bool(gap_exists and sig(c_att))}
    decisions[tgt] = flags
    for label, c in [("GAP: U dedicated − M4 own", c_gap), ("new head on M4 classifier input − M4 own", c_head), ("U dedicated − new head on M4 classifier input", c_rem),
                     ("new head on M4 pre − new head on M4 classifier input", c_att), ("U dedicated − best M4 representation + new head", c_best)]:
        dec_rows.append({"target": tgt, "gap_G_pp": G, "comparison": label, **c})
dec = pd.DataFrame(dec_rows); dec.to_csv(f"{OUT_DIR}/DEC_comparisons.csv", index=False)
print(dec[["target", "gap_G_pp", "comparison", "mean_diff_pp", "per_seed_pp", "video_lo_pp", "video_hi_pp", "seedvideo_lo_pp", "seedvideo_hi_pp"]].round(3).to_string(index=False))

print("\nFLAGS:", json.dumps(decisions))
rep_lim = any(f["representation"] for f in decisions.values())
no_gap = not any(f["gap_exists"] for f in decisions.values())
down_lim = any(f["downstream"] or f["attention"] for f in decisions.values())
if no_gap:
    verdict = "NO GAP IN THIS RUN: the dedicated models do not beat M4 here, so there is nothing to explain."
elif rep_lim:
    verdict = ("REPRESENTATION LIMITATION SUPPORTED. Next: run D3 (set RUN_D3=True) to check exposure, checkpoint selection and "
               "gradients before considering any balancing method; compare only against established remedies.")
elif down_lim:
    verdict = ("DOWNSTREAM / ATTENTION LIMITATION SUPPORTED, NO REPRESENTATION LIMITATION. Encoder-balancing methods are not "
               "justified by this evidence. Next: detailed comparison with missing-modality / missing-rate work (MissBench, MCE, "
               "EBMC, CMAD) before any proposal; D3 only if exposure/checkpoint selection must be separated.")
else:
    verdict = ("NO MECHANISM SUPPORTED. The gap is not isolated by these diagnostics. Stop this line; report it as a limitation.")
print("\nRESEARCH DECISION:", verdict)
json.dump({"flags": decisions, "verdict": verdict}, open(f"{OUT_DIR}/DEC_research_decision.json", "w"), indent=2)

target  gap_G_pp                                           comparison  mean_diff_pp         per_seed_pp  video_lo_pp  video_hi_pp  seedvideo_lo_pp  seedvideo_hi_pp
vision     2.406                            GAP: U dedicated − M4 own         2.406 +2.90, +3.00, +1.32        1.195        3.575            0.741            4.045
vision     2.406             new head on M4 classifier input − M4 own         2.941 +3.55, +2.22, +3.05        1.736        4.143            1.276            4.474
vision     2.406        U dedicated − new head on M4 classifier input        -0.535 -0.65, +0.78, -1.73       -1.453        0.390           -2.103            1.127
vision     2.406 new head on M4 pre − new head on M4 classifier input        -0.369 -0.08, -0.09, -0.94       -1.008        0.268           -1.394            0.541
vision     2.406      U dedicated − best M4 representation + new head        -0.535 -0.65, +0.78, -1.73       -1.453        0.390           -2.103            1.127
   A+V     1.983

## EMO (secondary). Is emotion supervision a plausible channel for the non-text modalities?
Runs only if every check in Section L passed. Question: relative to text, do audio/video predict the **emotion** labels
better than they predict **sentiment**? Measure: linear probes on raw pooled features (same probe and C selection as D1),
AUROC on test. Sentiment: one-vs-rest macro-AUROC over the 3 classes. Emotions: mean AUROC of 6 presence tasks
(intensity > 0). Relative signal r_m = (AUROC_m − 0.5) / (AUROC_text − 0.5).
**Decision rule (fixed in advance):** emotion supervision is considered a plausible targeted channel for modality m only if
r_m(emotions) ≥ 1.5 × r_m(sentiment). Even then, joint sentiment–emotion learning on MOSEI is established
(e.g. Akhtar et al., NAACL 2019), so only a specific, untested use would be a research opportunity.

In [14]:
if emo_ok and RUN_EMO:
    def probe_auc(Xtr, ytr, Xva, yva, Xte, yte, multiclass):
        best = None
        for C in C_GRID:
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)
            pv = clf.predict_proba(Xva)
            sc = roc_auc_score(yva, pv, multi_class="ovr", average="macro") if multiclass else roc_auc_score(yva, pv[:, 1])
            if best is None or sc > best[0]:
                best = (sc, clf)
        pt = best[1].predict_proba(Xte)
        return roc_auc_score(yte, pt, multi_class="ovr", average="macro") if multiclass else roc_auc_score(yte, pt[:, 1])
    keep = {sp: np.isin(np.asarray(data[sp]["id"]).astype(str), emo_table.index) for sp in SPLITS}
    print("clips used (matched):", {sp: int(k.sum()) for sp, k in keep.items()}, "| excluded (unmatched):", {sp: int((~k).sum()) for sp, k in keep.items()})
    emo_lab = {sp: emo_table.loc[np.asarray(data[sp]["id"]).astype(str)[keep[sp]]] for sp in SPLITS}
    ysent = {sp: data[sp]["y"][keep[sp]] for sp in SPLITS}
    feats_by = {"text": ["text"], "audio": ["audio"], "vision": ["vision"]}
    rows = []
    for m, mods in feats_by.items():
        X = {sp: raw_pooled(data[sp], mods)[keep[sp]] for sp in SPLITS}
        rows.append({"modality": m, "task": "sentiment (3-class)", "test_AUROC": probe_auc(X["train"], ysent["train"], X["valid"], ysent["valid"], X["test"], ysent["test"], True)})
        for e in EMO_NAMES:
            yy = {sp: (emo_lab[sp][e].values > 0).astype(int) for sp in SPLITS}
            rows.append({"modality": m, "task": f"emotion {e}", "test_AUROC": probe_auc(X["train"], yy["train"], X["valid"], yy["valid"], X["test"], yy["test"], False)})
    emo_res = pd.DataFrame(rows); emo_res.to_csv(f"{OUT_DIR}/EMO_probe_auroc.csv", index=False)
    summ = emo_res.assign(kind=np.where(emo_res.task.str.startswith("emotion"), "emotions (mean of 6)", "sentiment")).groupby(["modality", "kind"]).test_AUROC.mean().unstack()
    for kind in list(summ.columns):
        denom = summ.loc["text", kind] - 0.5
        summ[f"r_{kind}"] = (summ[kind] - 0.5) / denom if denom >= 0.02 else np.nan   # undefined if text itself is near chance
    summ["emotion_channel_plausible"] = (summ["r_emotions (mean of 6)"] >= 1.5 * summ["r_sentiment"]).astype(object)
    summ.loc["text", "emotion_channel_plausible"] = "reference"
    summ.to_csv(f"{OUT_DIR}/EMO_summary.csv")
    print(emo_res.pivot(index="task", columns="modality", values="test_AUROC").round(3).to_string())
    print(summ.round(3).to_string())
else:
    print("EMO skipped:", "RUN_EMO = False" if emo_ok else "emotion labels not verified (see Section L).")

EMO skipped: RUN_EMO = False


## D3 (optional, independently executable). Deterministic replay of M4 training with logging

**Run only if DEC says "representation limitation", or if exposure / checkpoint selection must be separated.**
To run it alone: run Sections 0–5, set `RUN_D3 = True`, then run this section.

The replay uses the **original v2 training procedure and random-number sequence**. Logging only *reads* values
(gradients before clipping, availability masks already drawn, extra evaluations in eval mode, which use no random numbers).
At the end the selected weights must hash **identically** to the v2 checkpoint; otherwise the replay is declared invalid.

Recorded per epoch: frequency of each modality combination in training batches; per-modality **encoder-projection**
gradient RMS (= ‖g‖ / √(number of parameters)), also divided by the share of batch samples in which that modality was
available; cross-attention gradients separately (they also depend on text); validation macro-F1 for every modality
combination; a unimodal score (mean probability of the true class on a fixed 2,048-clip training subsample with only that
modality available). Every epoch's weights are saved so alternative checkpoint-selection rules can be evaluated.
**Gradient magnitudes alone are not interpreted as proof of under-learning.**

Pre-stated reading for checkpoint selection: if choosing M4's epoch by **video-only validation F1** raises M4's video-only
test F1 by ≥ 50 % of the gap G (with the 3 seeds agreeing), checkpoint selection is a sufficient explanation of that much
of the gap.

In [15]:
SUBSET_ROWS = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}
SUBSET_KEY = {tuple(map(float, v)): k for k, v in SUBSET_ROWS.items()}

def weight_hash(state):
    return hashlib.md5(b"".join(v.numpy().tobytes() for v in state.values())).hexdigest()[:12]

@torch.no_grad()
def subset_eval(model, d, row):
    model.eval(); n = len(d["y"]); out = []
    for i in range(0, n, 256):
        idx = np.arange(i, min(i + 256, n))
        x, mask, av = to_batch(d, idx, np.tile(np.asarray(row, np.float32), (len(idx), 1)))
        out.append(model(x, mask, av).softmax(-1).numpy())
    return np.concatenate(out)

def replay_m4(seed, save_dir):
    """v2 train() for M4, unchanged in every computation, plus read-only logging."""
    name = "M4 Fixed fusion + modality dropout"; cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    probe_idx = np.random.RandomState(123).permutation(len(tr["y"]))[:2048]          # separate generator: no effect on training
    groups = {f"proj_{m}": list(model.proj[m].parameters()) for m in MODS}
    groups.update({f"xattn_{m}": list(model.xattn[m].parameters()) for m in ["audio", "vision"]})
    n_params = {g: sum(p.numel() for p in ps) for g, ps in groups.items()}
    best, best_state, bad, log, epoch_states = -1, None, 0, [], []
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        gsum = {g: 0.0 for g in groups}; gnorm_avail = {m: 0.0 for m in MODS}; nb = 0; combo = {k: 0 for k in SUBSET_ROWS}
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            avail_np = sample_avail(len(idx), (drop, drop, drop), rng)
            for r in avail_np: combo[SUBSET_KEY[tuple(map(float, r))]] += 1
            x, mask, avail = to_batch(tr, idx, avail_np)
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward()
            for g, ps in groups.items():                                              # read-only, before clipping
                rms = float(torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in ps if p.grad is not None)) / np.sqrt(n_params[g]))
                gsum[g] += rms
                if g.startswith("proj_"):
                    m = g[5:]; share = float(avail_np[:, MODS.index(m)].mean())
                    gnorm_avail[m] += rms / share if share > 0 else 0.0
            nb += 1
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        rec = {"seed": seed, "epoch": ep + 1, "valid_macro_f1_full": val}
        rec.update({f"grad_rms_{g}": v / nb for g, v in gsum.items()})
        rec.update({f"grad_rms_per_available_proj_{m}": v / nb for m, v in gnorm_avail.items()})
        tot = sum(combo.values()); rec.update({f"share_{k}": c / tot for k, c in combo.items()})
        for k, row in SUBSET_ROWS.items():
            rec[f"valid_f1_{k}"] = mf1(y_val, subset_eval(model, data["valid"], row).argmax(1))
        sub = {"feats": {m: tr["feats"][m][probe_idx] for m in MODS}, "mask": tr["mask"][probe_idx], "y": tr["y"][probe_idx]}
        for k in ["T", "A", "V"]:
            p = subset_eval(model, sub, SUBSET_ROWS[k]); rec[f"train_true_class_prob_{k}"] = float(p[np.arange(len(sub["y"])), sub["y"]].mean())
        log.append(rec)
        st = {k: v.detach().clone() for k, v in model.state_dict().items()}; epoch_states.append(st)
        torch.save(st, f"{save_dir}/M4_seed{seed}_epoch{ep + 1:02d}.pt")
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(log), epoch_states

if RUN_D3:
    D3_DIR = f"{OUT_DIR}/D3_replay"; os.makedirs(D3_DIR, exist_ok=True)
    G_v = None
    rows_sel, logs_all, hash_rows = [], [], []
    for s in D3_SEEDS:
        t0 = time.time()
        model, lg, states = replay_m4(s, D3_DIR)
        h_new, h_v2 = weight_hash(model.state_dict()), weight_hash(torch.load(f"{CKPT_DIR}/M4_seed{s}.pt", map_location="cpu"))
        hash_rows.append({"seed": s, "replay_hash": h_new, "v2_hash": h_v2, "identical": h_new == h_v2, "seconds": round(time.time() - t0, 1)})
        logs_all.append(lg)
        print(f"seed {s}: replay hash {h_new} vs v2 {h_v2} ->", "IDENTICAL" if h_new == h_v2 else "DIFFERENT (replay invalid)")
        for rule, col in [("v2 rule: full-modality valid F1", "valid_macro_f1_full"), ("video-only valid F1", "valid_f1_V"),
                          ("A+V-only valid F1", "valid_f1_A+V")]:
            ep = int(lg.loc[lg[col].idxmax(), "epoch"]); m = FixedFusion(); m.load_state_dict(states[ep - 1])
            rows_sel.append({"seed": s, "selection_rule": rule, "epoch": ep,
                             "test_f1_V_only": mf1(y_test, subset_eval(m, data["test"], SUBSET_ROWS["V"]).argmax(1)),
                             "test_f1_AV_only": mf1(y_test, subset_eval(m, data["test"], SUBSET_ROWS["A+V"]).argmax(1)),
                             "test_f1_full": mf1(y_test, subset_eval(m, data["test"], SUBSET_ROWS["T+A+V"]).argmax(1))})
    pd.DataFrame(hash_rows).to_csv(f"{OUT_DIR}/D3_hash_check.csv", index=False)
    pd.concat(logs_all).to_csv(f"{OUT_DIR}/D3_epoch_log.csv", index=False)
    sel_df = pd.DataFrame(rows_sel); sel_df.to_csv(f"{OUT_DIR}/D3_checkpoint_selection.csv", index=False)
    print(pd.DataFrame(hash_rows).to_string(index=False))
    if not all(r["identical"] for r in hash_rows):
        print("REPLAY INVALID for at least one seed: do not interpret D3 results for that seed.")
    print(sel_df.groupby("selection_rule")[["epoch", "test_f1_V_only", "test_f1_AV_only", "test_f1_full"]].mean().round(4))
    # Pre-stated reading: does selecting by video-only validation F1 recover >= 50 % of the video gap G (all seeds agreeing)?
    g_rows = []
    for s in D3_SEEDS:
        v2r = sel_df[(sel_df.seed == s) & (sel_df.selection_rule == "v2 rule: full-modality valid F1")].iloc[0]
        vr = sel_df[(sel_df.seed == s) & (sel_df.selection_rule == "video-only valid F1")].iloc[0]
        u2 = mf1(y_test, np.load(f"{V4_DIR}/checkpoints/U2_seed{s}_test_probs.npy").argmax(1))
        g_rows.append({"seed": s, "G_pp": 100 * (u2 - v2r.test_f1_V_only), "gain_from_video_selection_pp": 100 * (vr.test_f1_V_only - v2r.test_f1_V_only)})
    g_df = pd.DataFrame(g_rows); g_df.to_csv(f"{OUT_DIR}/D3_selection_verdict.csv", index=False)
    G_mean, gain_mean = g_df.G_pp.mean(), g_df.gain_from_video_selection_pp.mean()
    explains = G_mean > 0 and gain_mean >= 0.5 * G_mean and (g_df.gain_from_video_selection_pp > 0).all()
    print(g_df.round(3).to_string(index=False))
    print(f"Video gap G = {G_mean:+.2f} pp; gain from video-based checkpoint selection = {gain_mean:+.2f} pp ->",
          "checkpoint selection explains at least half of the video gap" if explains else "checkpoint selection does NOT explain half of the video gap")
    p = MOD_DROPOUT
    print(f"Expected shares with p={p} (if all three are dropped, one is restored at random): "
          f"V only {p**2*(1-p) + p**3/3:.4f}, A only {p**2*(1-p) + p**3/3:.4f}, A+V only {p*(1-p)**2:.4f}, "
          f"text missing (any) {p - p**3/3:.4f}, all three present {(1-p)**3:.4f}")
    print("Observed shares in the replayed training batches (mean over epochs and seeds):")
    print(pd.concat(logs_all)[[c for c in logs_all[0].columns if c.startswith("share_")]].mean().round(4).to_string())
else:
    print("D3 not run (RUN_D3 = False).")

D3 not run (RUN_D3 = False).


## Files written (v1–v4 folders untouched)

In [16]:
for root, _, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        if not fn.endswith(".pt"):
            print("  ", os.path.relpath(os.path.join(root, fn), OUT_DIR))

   D1_linear_probes_per_seed.csv
   D2_heads_per_seed.csv
   DEC_comparisons.csv
   DEC_research_decision.json
   LF_M1_U3_seed42_test_probs.npy
   LF_M1_U3_seed43_test_probs.npy
   LF_M1_U3_seed44_test_probs.npy
   R_reference_baselines.csv
   R_reference_comparisons.csv
   label_verification.json
   mosei_emotion_labels_matched.csv
   run_info.json
   test_ids.csv
   test_labels.npy
   test_scores.npy
   valid_labels.npy
   D1_linear_parts/M4_fused_A_V_seed42.json
   D1_linear_parts/M4_fused_A_V_seed42_pred.npy
   D1_linear_parts/M4_fused_A_V_seed43.json
   D1_linear_parts/M4_fused_A_V_seed43_pred.npy
   D1_linear_parts/M4_fused_A_V_seed44.json
   D1_linear_parts/M4_fused_A_V_seed44_pred.npy
   D1_linear_parts/M4_post_alone_audio_seed42.json
   D1_linear_parts/M4_post_alone_audio_seed42_pred.npy
   D1_linear_parts/M4_post_alone_audio_seed43.json
   D1_linear_parts/M4_post_alone_audio_seed43_pred.npy
   D1_linear_parts/M4_post_alone_audio_seed44.json
   D1_linear_parts/M4_post_alone_a